In [1]:
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

assert torch.cuda.is_available(), "Нужна CUDA"

torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

dev = "cuda"
eps = 1e-5

In [2]:
fwd_cfgs = [
    triton.Config({"BS": 128}, num_warps=2),
    triton.Config({"BS": 256}, num_warps=4),
    triton.Config({"BS": 512}, num_warps=8),
    triton.Config({"BS": 1024}, num_warps=8),
]

@triton.autotune(configs=fwd_cfgs, key=["N"])
@triton.jit
def ln_fwd(xp, yp, wp, bp, mp, rp, st, N, eps, BS: tl.constexpr):
    row = tl.program_id(0)
    xp += row * st
    yp += row * st

    sm = tl.zeros([BS], dtype=tl.float32)
    for off in range(0, N, BS):
        idx = off + tl.arange(0, BS)
        m = idx < N
        x = tl.load(xp + idx, mask=m, other=0.).to(tl.float32)
        sm += x
    mu = tl.sum(sm, axis=0) / N

    sv = tl.zeros([BS], dtype=tl.float32)
    for off in range(0, N, BS):
        idx = off + tl.arange(0, BS)
        m = idx < N
        x = tl.load(xp + idx, mask=m, other=0.).to(tl.float32)
        x = tl.where(m, x - mu, 0.)
        sv += x * x
    var = tl.sum(sv, axis=0) / N
    rs = 1.0 / tl.sqrt(var + eps)

    tl.store(mp + row, mu)
    tl.store(rp + row, rs)

    for off in range(0, N, BS):
        idx = off + tl.arange(0, BS)
        m = idx < N
        w = tl.load(wp + idx, mask=m, other=0.).to(tl.float32)
        b = tl.load(bp + idx, mask=m, other=0.).to(tl.float32)
        x = tl.load(xp + idx, mask=m, other=0.).to(tl.float32)
        y = (x - mu) * rs * w + b
        tl.store(yp + idx, y, mask=m)

def ln_fwd_run(x, w, b, eps=1e-5):
    x = x.contiguous()
    w = w.contiguous()
    b = b.contiguous()

    x2 = x.reshape(-1, x.shape[-1]).contiguous()
    m, n = x2.shape

    y2 = torch.empty_like(x2)
    mu = torch.empty(m, device=x.device, dtype=torch.float32)
    rs = torch.empty(m, device=x.device, dtype=torch.float32)

    ln_fwd[(m,)](x2, y2, w, b, mu, rs, x2.stride(0), n, eps)
    return y2.reshape_as(x), mu, rs

In [3]:
fwd_cfgs = [
    triton.Config({"BS": 128}, num_warps=2),
    triton.Config({"BS": 256}, num_warps=4),
    triton.Config({"BS": 512}, num_warps=8),
    triton.Config({"BS": 1024}, num_warps=8),
]

@triton.autotune(configs=fwd_cfgs, key=["N"])
@triton.jit
def ln_fwd(xp, yp, wp, bp, mp, rp, st, N, eps, BS: tl.constexpr):
    row = tl.program_id(0)
    xp += row * st
    yp += row * st

    sm = tl.zeros([BS], dtype=tl.float32)
    for off in range(0, N, BS):
        idx = off + tl.arange(0, BS)
        m = idx < N
        x = tl.load(xp + idx, mask=m, other=0.).to(tl.float32)
        sm += x
    mu = tl.sum(sm, axis=0) / N

    sv = tl.zeros([BS], dtype=tl.float32)
    for off in range(0, N, BS):
        idx = off + tl.arange(0, BS)
        m = idx < N
        x = tl.load(xp + idx, mask=m, other=0.).to(tl.float32)
        x = tl.where(m, x - mu, 0.)
        sv += x * x
    var = tl.sum(sv, axis=0) / N
    rs = 1.0 / tl.sqrt(var + eps)

    tl.store(mp + row, mu)
    tl.store(rp + row, rs)

    for off in range(0, N, BS):
        idx = off + tl.arange(0, BS)
        m = idx < N
        w = tl.load(wp + idx, mask=m, other=0.).to(tl.float32)
        b = tl.load(bp + idx, mask=m, other=0.).to(tl.float32)
        x = tl.load(xp + idx, mask=m, other=0.).to(tl.float32)
        y = (x - mu) * rs * w + b
        tl.store(yp + idx, y, mask=m)

def ln_fwd_run(x, w, b, eps=1e-5):
    x = x.contiguous()
    w = w.contiguous()
    b = b.contiguous()

    x2 = x.reshape(-1, x.shape[-1]).contiguous()
    m, n = x2.shape

    y2 = torch.empty_like(x2)
    mu = torch.empty(m, device=x.device, dtype=torch.float32)
    rs = torch.empty(m, device=x.device, dtype=torch.float32)

    ln_fwd[(m,)](x2, y2, w, b, mu, rs, x2.stride(0), n, eps)
    return y2.reshape_as(x), mu, rs

In [4]:
def chk_fwd():
    for m, n in [(4, 256), (8, 1024), (16, 1536)]:
        x = torch.randn(m, n, device=dev, dtype=torch.float32)
        w = torch.randn(n, device=dev, dtype=torch.float32)
        b = torch.randn(n, device=dev, dtype=torch.float32)

        y0 = F.layer_norm(x, (n,), w, b, eps)
        y1, _, _ = ln_fwd_run(x, w, b, eps)

        torch.testing.assert_close(y1, y0, atol=1e-4, rtol=1e-4)

    print("forward OK")

chk_fwd()

forward OK


In [5]:
bwd_cfgs = [
    triton.Config({}, num_warps=2),
    triton.Config({}, num_warps=4),
    triton.Config({}, num_warps=8),
]

@triton.autotune(configs=bwd_cfgs, key=["N"], reset_to_zero=["dwp", "dbp"])
@triton.jit
def ln_bwd(dx, dy, dwp, dbp, x, w, mu, rs, st, N, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    x += row * st
    dy += row * st
    dx += row * st

    mu = tl.load(mu + row).to(tl.float32)
    rs = tl.load(rs + row).to(tl.float32)

    idx = tl.arange(0, BLOCK_N)
    m = idx < N

    xv = tl.load(x + idx, mask=m, other=0.).to(tl.float32)
    gy = tl.load(dy + idx, mask=m, other=0.).to(tl.float32)
    ww = tl.load(w + idx, mask=m, other=0.).to(tl.float32)

    xh = (xv - mu) * rs
    wdy = ww * gy

    xh = tl.where(m, xh, 0.)
    wdy = tl.where(m, wdy, 0.)

    c1 = tl.sum(xh * wdy, axis=0) / N
    c2 = tl.sum(wdy, axis=0) / N

    dxi = (wdy - (xh * c1 + c2)) * rs
    tl.store(dx + idx, dxi, mask=m)

    tl.atomic_add(dwp + idx, gy * xh, mask=m)
    tl.atomic_add(dbp + idx, gy, mask=m)


def ln_bwd_run(x, dy, w, mu, rs):
    x = x.contiguous()
    dy = dy.contiguous()
    w = w.contiguous()

    x2 = x.reshape(-1, x.shape[-1]).contiguous()
    dy2 = dy.reshape_as(x2).contiguous()
    m_dim, n_dim = x2.shape

    dx2 = torch.empty_like(x2)
    dw = torch.zeros(n_dim, device=x.device, dtype=torch.float32)
    db = torch.zeros(n_dim, device=x.device, dtype=torch.float32)

    BLOCK_N = triton.next_power_of_2(n_dim)

    ln_bwd[(m_dim,)](dx2, dy2, dw, db, x2, w, mu, rs, x2.stride(0), n_dim, BLOCK_N=BLOCK_N)

    return dx2.reshape_as(x), dw, db

In [6]:
class TritonLayerNorm(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, w, b, eps=1e-5):
        y, mu, rs = ln_fwd_run(x, w, b, eps)
        ctx.save_for_backward(x, w, b, mu, rs)
        return y

    @staticmethod
    def backward(ctx, dy):
        x, w, b, mu, rs = ctx.saved_tensors
        dx, dw, db = ln_bwd_run(x, dy, w, mu, rs)
        return dx, dw.to(w.dtype), db.to(b.dtype), None

layer_norm = TritonLayerNorm.apply

def chk_bwd():
    for m, n in [(8, 256), (16, 1024), (4, 1536)]:
        x = torch.randn(m, n, device=dev, dtype=torch.float32, requires_grad=True)
        w = torch.randn(n, device=dev, dtype=torch.float32, requires_grad=True)
        b = torch.randn(n, device=dev, dtype=torch.float32, requires_grad=True)
        dy = torch.randn_like(x)

        y0 = F.layer_norm(x, (n,), w, b, eps)
        y1 = layer_norm(x, w, b, eps)

        torch.testing.assert_close(y1, y0, atol=1e-4, rtol=1e-4)

        y1.backward(dy, retain_graph=True)
        dx1, dw1, db1 = x.grad.clone(), w.grad.clone(), b.grad.clone()

        x.grad = None
        w.grad = None
        b.grad = None

        y0.backward(dy)
        torch.testing.assert_close(dx1, x.grad, atol=1e-3, rtol=1e-3)
        torch.testing.assert_close(dw1, w.grad, atol=1e-3, rtol=1e-3)
        torch.testing.assert_close(db1, b.grad, atol=1e-3, rtol=1e-3)

    print("backward OK")

chk_bwd()

backward OK


In [7]:
def bench(m=2048, n=1024, dtype=torch.float32, eps=1e-5):
    x = torch.randn(m, n, device=dev, dtype=dtype)
    w = torch.randn(n, device=dev, dtype=dtype)
    b = torch.randn(n, device=dev, dtype=dtype)
    dy = torch.randn_like(x)

    _ = layer_norm(x, w, b, eps)

    def tri_fwd():
        layer_norm(x, w, b, eps)

    def torch_fwd():
        F.layer_norm(x, (n,), w, b, eps)

    x1 = x.clone().requires_grad_(True)
    w1 = w.clone().requires_grad_(True)
    b1 = b.clone().requires_grad_(True)

    def tri_bwd():
        x1.grad = None
        w1.grad = None
        b1.grad = None
        y = layer_norm(x1, w1, b1, eps)
        y.backward(dy)

    x2 = x.clone().requires_grad_(True)
    w2 = w.clone().requires_grad_(True)
    b2 = b.clone().requires_grad_(True)

    def torch_bwd():
        x2.grad = None
        w2.grad = None
        b2.grad = None
        y = F.layer_norm(x2, (n,), w2, b2, eps)
        y.backward(dy)

    tf = triton.testing.do_bench(tri_fwd, return_mode="median")
    rf = triton.testing.do_bench(torch_fwd, return_mode="median")
    tb = triton.testing.do_bench(tri_bwd, return_mode="median")
    rb = triton.testing.do_bench(torch_bwd, return_mode="median")

    print(f"Forward : Triton {tf:.3f} ms | Torch {rf:.3f} ms | speedup {rf / tf:.2f}x")
    print(f"Backward: Triton {tb:.3f} ms | Torch {rb:.3f} ms | speedup {rb / tb:.2f}x")

bench()

Forward : Triton 0.073 ms | Torch 0.079 ms | speedup 1.07x
Backward: Triton 0.281 ms | Torch 0.285 ms | speedup 1.02x
